# 07. Sampling: превращаем velocity в траекторию

Разберём Euler, shifted schedule, classifier-free guidance, distillation и NFE. Сначала точная простая система, затем связь с generate/get_velocity.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/07_sampling.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Одна строка sampler

Euler говорит: новое положение = текущее + длительность шага × скорость. В Kandinsky timestep уменьшается, поэтому шаг отрицательный. Покажем постоянную velocity, где Euler точен даже с одним шагом.


In [ ]:
noise=torch.tensor([1.5,-0.3]); data=torch.tensor([0.2,0.7]); v=noise-data
x=noise.clone(); schedule=shifted_schedule(16,5)
trajectory=[x.clone()]
for dt in schedule.diff(): x=x+dt*v; trajectory.append(x.clone())
print('end:',x,'expected:',data)
assert torch.allclose(x,data,atol=1e-6)


### 2. Shifted schedule

t′=s·t/(1+(s−1)t). s меняет плотность временных точек; начальная и конечная точки остаются 1 и 0. YAML distill содержит scheduler_scale=5, но pipeline.__call__ по умолчанию использует 10. В курсе передаём 5 явно и сохраняем его в отчёт; можно сравнить 5/10 как эксперимент.


In [ ]:
for scale in [1,3,5,10]:
    times=shifted_schedule(16,scale)
    plt.plot(range(17),times,marker='.',label=f's={scale}')
plt.xlabel('step'); plt.ylabel('time'); plt.legend(); plt.show()
assert torch.all(schedule.diff()<0)
print('sum Δt:',schedule.diff().sum().item())


### 3. Нелинейная velocity и ошибка Euler

На постоянном поле больше шагов ничего не улучшает. Для нелинейного по времени поля ошибка уже зависит от шага. Здесь dx/dt=x, x(1)=1, точный ответ x(0)=e⁻¹. Это учебная ODE, не Kandinsky velocity.


In [ ]:
errors=[]
for steps in [4,8,16,32,64]:
    x=torch.tensor(1.); times=shifted_schedule(steps,1)
    for dt in times.diff(): x=x+dt*x
    error=abs(x.item()-np.exp(-1)); errors.append(error)
    print(steps,x.item(),error)
plt.loglog([4,8,16,32,64],errors,'o-'); plt.xlabel('Euler steps'); plt.ylabel('absolute error'); plt.show()


### 4. CFG требует двух вызовов

v=v_uncond+w·(v_cond−v_uncond). При w=1 это v_cond: upstream пропускает uncond forward. Distill checkpoint научился использовать одну ветвь. Guidance=1 здесь не означает отсутствие text conditioning.


In [ ]:
cond=torch.tensor([2.,1.]); uncond=torch.tensor([0.,-1.])
for w in [0.,1.,3.,5.]: print(w,uncond+w*(cond-uncond))
for label,steps,w in [('SFT',50,5),('noCFG',50,1),('distill',16,1)]:
    print(label,'steps',steps,'NFE',steps*(1 if w==1 else 2))


### 5. Distillation не уменьшает архитектуру

SFT, noCFG и distilled16steps имеют одинаковые основные размеры DiT. Различаются обученные веса и рекомендуемый режим. Поставить num_steps=16 на SFT — не то же самое, что загрузить distill. Heun требует дополнительный forward; он здесь учебный, не официальная рекомендация для 16-step checkpoint.


In [ ]:
from omegaconf import OmegaConf
for suffix in ['sft','nocfg','distil']:
    cfg=OmegaConf.load(SOURCE/f'configs/k5_lite_t2v_5s_{suffix}_sd.yaml')
    print(suffix,'steps',cfg.model.num_steps,'guidance',cfg.model.guidance_weight,'width',cfg.model.dit_params.model_dim)


### 6. Связываем с кодом

get_velocity передаёт t×1000 в DiT и при необходимости делает CFG. generate создаёт timesteps, visual conditioning и Euler update. Для T2V first_frames=None. Для I2V первый латент фиксируется: это дополнительное условие, не смена text tokenizer.


In [ ]:
source_excerpt('kandinsky/generation_utils.py','def get_velocity',49)
source_excerpt('kandinsky/generation_utils.py','timesteps = torch.linspace',47)


### 7. Протокол честного сравнения

Меняем один фактор: seed, steps, scale или checkpoint. Для latency отделяем загрузку, text encoding, sampling, decoding и запись файла. GPU вычисляет асинхронно: синхронизация перед/после замера обязательна. Не сравниваем холодный первый запуск с прогретым.


In [ ]:
from labkit.common import save_json
experiment={'checkpoint':'distilled16steps-5s','seed':42,'steps':16,'NFE':16,'guidance':1.,'scheduler_scale':5.,'height':512,'width':768,'source_revision':REVISION,'measured':False}
save_json(ROOT/'artifacts/sampling_protocol.json',experiment)
print(experiment)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
